# IM-RoTHP: Operação Resgate (Estabilização e Grid Search)

Este notebook testa uma versão estabilizada do **IM-RoTHP** (Intensity-Modulated Rotary THP).

### Mudanças em relação à versão anterior:
1.  **Modulação Suave (Soft Gating):** Usamos `tanh` para limitar a distorção máxima do tempo a 10% (`max_modulation=0.1`).
2.  **Inicialização Cautelosa:** O parâmetro `alpha` começa em 0.0 (sem modulação) e é aprendido gradualmente.
3.  **Teste Multi-Dataset:** Avaliamos no `retweet` (caótico) e `stackoverflow` (estruturado).

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch FP16
import easy_tpp.model.torch_model.torch_baselayer as baselayer
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

from easy_tpp.model.torch_model.torch_rothp import RoTHP, RotaryEmbedding
from easy_tpp.model.torch_model.torch_imrothp import IMRoTHP, IntensityModulatedRotaryEmbedding

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")

### 1. Definição da Classe Estabilizada

In [ ]:
# --- NOVA VERSÃO ESTABILIZADA DO IM-RoPE ---
class StabilizedIMRotaryEmbedding(RotaryEmbedding):
    def __init__(self, dim, max_freq=10000, max_modulation=0.1):
        super().__init__(dim, max_freq)
        self.alpha = nn.Parameter(torch.tensor(0.0)) # Começa DESLIGADO (Alpha=0)
        self.max_modulation = max_modulation # Limite hard de distorção (ex: 10%)

    def forward(self, t, intensity):
        # Modulação suave e limitada
        # theta_new = theta * (1 + max_mod * tanh(alpha * log(1+lambda)))
        # Se alpha=0, modulação=0 -> Comportamento idêntico ao RoTHP
        
        # [batch, seq, 1]
        raw_mod = self.alpha * torch.log1p(intensity.unsqueeze(-1))
        modulation = 1.0 + self.max_modulation * torch.tanh(raw_mod)
        
        t_expanded = t.unsqueeze(-1)
        thetas_expanded = self.thetas.view(1, 1, -1)
        
        args = t_expanded * thetas_expanded * modulation
        
        cos_args = torch.cos(args)
        sin_args = torch.sin(args)
        
        # Interleave
        cos = torch.repeat_interleave(cos_args, 2, dim=-1)
        sin = torch.repeat_interleave(sin_args, 2, dim=-1)
        return cos, sin

# Sobrescrever a classe no modelo
class StabilizedIMRoTHP(IMRoTHP):
    def __init__(self, model_config):
        super().__init__(model_config)
        # Substituir pelo embedding estabilizado
        self.rotary_emb = StabilizedIMRotaryEmbedding(self.d_model // self.n_head)

### 2. Configuração e Datasets

In [ ]:
class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = None

def get_dataloader(ds_name):
    print(f"Carregando {ds_name}...")
    dataset = load_dataset(f"easytpp/{ds_name}")
    train_data = dataset['train']
    dev_data = dataset['validation']
    
    all_deltas = []
    for item in train_data:
        all_deltas.extend([d for d in item['time_since_last_event'] if d > 0])
    time_scale = np.mean(all_deltas)
    
    # Num types
    max_type = 0
    for x in train_data:
        if len(x['type_event']) > 0:
            max_type = max(max_type, max(x['type_event']))
    num_types = max_type + 1
    pad_id = num_types
    
    # Collate local para simplificar
    def collate_fn(batch_list):
        batch_size = len(batch_list)
        max_len = max(len(x['time_since_start']) for x in batch_list)
        pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
        pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
        pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
        batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
        attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
        causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
        for i, item in enumerate(batch_list):
            l = len(item['time_since_start'])
            ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
            td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
            ev = torch.tensor(item['type_event'], dtype=torch.long)
            ts = (ts - ts[0]) / time_scale
            td = td / time_scale
            pad_time[i, :l] = ts.float()
            pad_delta[i, :l] = td.float()
            pad_type[i, :l] = ev
            batch_non_pad_mask[i, :l] = 1.0
            mask_i = causal_mask_base.clone()
            mask_i[:, l:] = True
            mask_i[l:, :] = True
            attention_mask[i] = mask_i
        return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

    train_loader = DataLoader(train_data, batch_size=1024, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_data, batch_size=256, shuffle=False, collate_fn=collate_fn)
    
    return train_loader, dev_loader, num_types, pad_id

### 3. Experimento Comparativo (Loop)

In [ ]:
datasets_to_test = ['retweet', 'stackoverflow']
results = []

for ds_name in datasets_to_test:
    train_l, dev_l, num_types, pad_id = get_dataloader(ds_name)
    config = ModelConfig(num_types, pad_id)
    
    # 1. Treinar Baseline (RoTHP)
    print(f"[{ds_name}] Treinando RoTHP Baseline...")
    model_base = RoTHP(config).to(device)
    optim = torch.optim.AdamW(model_base.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    best_nll_base = float('inf')
    for ep in range(15): # 15 épocas rápidas
        model_base.train()
        for batch in train_l:
            batch = [t.to(device) for t in batch]
            optim.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model_base.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
            
        # Val
        model_base.eval()
        val_loss = 0
        val_num = 0
        with torch.no_grad():
            for batch in dev_l:
                batch = [t.to(device) for t in batch]
                with torch.amp.autocast('cuda'):
                    l, n = model_base.loglike_loss(batch)
                val_loss += l.item()
                val_num += n
        nll = val_loss / (val_num + 1e-9)
        best_nll_base = min(best_nll_base, nll)
        print(f"  Ep {ep}: NLL {nll:.4f}")

    # 2. Treinar Proposta (Stabilized IM-RoTHP)
    print(f"[{ds_name}] Treinando Stabilized IM-RoTHP...")
    model_im = StabilizedIMRoTHP(config).to(device)
    optim = torch.optim.AdamW(model_im.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    best_nll_im = float('inf')
    for ep in range(15):
        model_im.train()
        for batch in train_l:
            batch = [t.to(device) for t in batch]
            optim.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model_im.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
    
        # Val
        model_im.eval()
        val_loss = 0
        val_num = 0
        with torch.no_grad():
            for batch in dev_l:
                batch = [t.to(device) for t in batch]
                with torch.amp.autocast('cuda'):
                    l, n = model_im.loglike_loss(batch)
                val_loss += l.item()
                val_num += n
        nll = val_loss / (val_num + 1e-9)
        best_nll_im = min(best_nll_im, nll)
        
        # Verificar Alpha
        alpha = model_im.rotary_emb.alpha.item()
        print(f"  Ep {ep}: NLL {nll:.4f} | Alpha: {alpha:.6f}")

    results.append({
        'Dataset': ds_name,
        'Baseline NLL': best_nll_base,
        'IM-RoTHP NLL': best_nll_im,
        'Final Alpha': alpha,
        'Improvement': best_nll_base - best_nll_im
    })

print("\n--- RESULTADOS FINAIS ---")
print(pd.DataFrame(results))